# Fase 1 — KPIs Base
Portfolio 44 apartamentos turísticos · Bilbao · 2019-2025

Este notebook ejecuta `scripts/kpis_phase1.py` y muestra los resultados interactivos.

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / 'scripts'))
os.chdir(ROOT)

import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = 'notebook'

# Inventario real por fecha de apertura (detectado de los datos)
INVENTARIO = {
    'EDIFICIO_A':       2,
    'EDIFICIO_B':     9,
    'EDIFICIO_C':    9,
    'EDIFICIO_D': 7,
    'EDIFICIO_E':   14,
}
TOTAL_UNITS = sum(INVENTARIO.values())
OPENING_DATES = {}

df_all = pd.read_parquet('data/processed/reservas_unified.parquet')
df_all['check_in'] = pd.to_datetime(df_all['check_in'])
df_all['check_out'] = pd.to_datetime(df_all['check_out'])

# Fechas de apertura reales por edificio
OPENING_DATES = df_all[~df_all['cancelled']].groupby('building')['check_in'].min().to_dict()
print('Fechas de apertura detectadas:')
for b, d in sorted(OPENING_DATES.items(), key=lambda x: x[1]):
    print(f'  {b}: {d.date()}')

df = df_all[
    (~df_all['cancelled']) &
    (df_all['channel'] != 'Blocked channel') &
    (df_all['check_in'].dt.year.between(2019, 2025))
].copy()
print(f'\nReservas activas 2019-2025: {len(df):,}')

: 

## 1 · Revenue Anual por Canal

In [ ]:
CHANNEL_COLORS = {
    'Booking.com': '#003580', 'Airbnb': '#FF5A5F',
    'Direct booking': '#00A699', 'Website': '#FC642D', 'manual': '#999999',
}
yearly_channel = df.groupby(['year','channel'])['gross_amount'].sum().reset_index()
yearly_total   = df.groupby('year')['gross_amount'].sum().reset_index()

fig = px.bar(yearly_channel, x='year', y='gross_amount', color='channel',
             color_discrete_map=CHANNEL_COLORS, barmode='stack',
             title='Revenue Bruto Anual por Canal (2019-2025)',
             labels={'gross_amount':'Revenue bruto (€)','year':'Año','channel':'Canal'},
             template='plotly_white')
for _, r in yearly_total.iterrows():
    fig.add_annotation(x=r['year'], y=r['gross_amount'],
                       text=f"€{r['gross_amount']/1e6:.2f}M",
                       showarrow=False, yshift=12, font=dict(size=11))
fig.update_layout(legend=dict(orientation='h', y=-0.15))
fig.show()

## 2 · ADR y RevPAR Mensual

In [ ]:
def avail_nights_period(year, month):
    days = pd.Period(f'{year}-{month:02d}').days_in_month
    # Solo contar unidades abiertas ese mes
    period_start = pd.Timestamp(year, month, 1)
    units = sum(v for b, v in INVENTARIO.items()
                if OPENING_DATES.get(b, period_start) <= period_start)
    return units * days

monthly = df.groupby(['year','month']).agg(
    revenue=('gross_amount','sum'), nights_sold=('nights','sum')
).reset_index()
monthly['ADR']    = (monthly['revenue'] / monthly['nights_sold']).round(2)
monthly['avail']  = monthly.apply(lambda r: avail_nights_period(int(r['year']),int(r['month'])), axis=1)
monthly['RevPAR'] = (monthly['revenue'] / monthly['avail']).round(2)
monthly['period'] = pd.to_datetime(monthly[['year','month']].assign(day=1))

fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_trace(go.Scatter(x=monthly['period'], y=monthly['ADR'],
    name='ADR', line=dict(color='#003580', width=2)), secondary_y=False)
fig.add_trace(go.Scatter(x=monthly['period'], y=monthly['RevPAR'],
    name='RevPAR', line=dict(color='#FF5A5F', width=2, dash='dash')), secondary_y=True)
fig.update_layout(title='ADR y RevPAR Mensual (2019-2025)', template='plotly_white',
                  legend=dict(orientation='h', y=-0.15))
fig.update_yaxes(title_text='ADR (€)', secondary_y=False)
fig.update_yaxes(title_text='RevPAR (€)', secondary_y=True)
fig.show()

## 3 · Heatmap Ocupación %

In [ ]:
occ_data = []
for year in range(2019, 2026):
    for month in range(1, 13):
        mask = (df['check_in'].dt.year == year) & (df['check_in'].dt.month == month)
        nights_sold = df.loc[mask, 'nights'].sum()
        avail = avail_nights_period(year, month)
        occ_data.append({'year': year, 'month': month,
                         'occ_pct': round(nights_sold / avail * 100, 1) if avail > 0 else 0})
occ_pivot = pd.DataFrame(occ_data).pivot('year','month','occ_pct')
occ_pivot.columns = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']

fig = px.imshow(occ_pivot, color_continuous_scale='RdYlGn', zmin=0, zmax=100,
                title='Ocupación % por Mes y Año (portfolio completo)',
                labels=dict(x='Mes', y='Año', color='Ocupación %'),
                template='plotly_white', text_auto='.0f', aspect='auto')
fig.show()

## 4 · ALOS por Año y Canal

In [ ]:
alos = df.groupby(['year','channel'])['nights'].mean().reset_index()
alos.columns = ['year','channel','ALOS']
fig = px.line(alos[alos['channel'].isin(['Booking.com','Airbnb','Direct booking'])],
              x='year', y='ALOS', color='channel', markers=True,
              color_discrete_map=CHANNEL_COLORS,
              title='ALOS por Año y Canal',
              labels={'ALOS':'Noches medias','year':'Año'},
              template='plotly_white')
fig.update_xaxes(tickmode='linear', dtick=1)
fig.show()

## 5 · Distribución Booking Window

In [ ]:
lt = df[df['lead_time_days'].between(0,365)].copy()
BINS   = [0,1,3,7,14,30,60,90,180,366]
LABELS = ['Same day','1-3d','4-7d','8-14d','15-30d','31-60d','61-90d','91-180d','180d+']
lt['bucket'] = pd.cut(lt['lead_time_days'], bins=BINS, labels=LABELS, right=False)
lt_counts = lt['bucket'].value_counts().reindex(LABELS).reset_index()
lt_counts.columns = ['bucket','count']
lt_counts['pct'] = (lt_counts['count'] / lt_counts['count'].sum() * 100).round(1)
fig = px.bar(lt_counts, x='bucket', y='count',
             text=lt_counts['pct'].apply(lambda x: f'{x}%'),
             title='Distribución Booking Window (Lead Time)',
             template='plotly_white', color='count', color_continuous_scale='Blues')
fig.update_traces(textposition='outside')
fig.update_layout(showlegend=False, coloraxis_showscale=False)
fig.show()

## 6-10 · Resto de figuras

In [ ]:
from IPython.display import IFrame, display
from pathlib import Path
import os

figs = [
    '01_mix_canales_anual', '01_cancellation_rate',
    '01_adr_dia_semana', '01_net_adr_canal', '01_revenue_edificio_anual'
]
for f in figs:
    path = Path('../outputs/figures') / f'{f}.html'
    print(f'Abre en el navegador: {path.resolve()}')

## Tabla resumen KPIs
Ver: `outputs/reports/01_kpis_summary.md`